https://github.com/Midvel/medium_jupyter_notes/blob/master/naive_bayes_filter/bayes-classificator.ipynb

https://towardsdatascience.com/how-to-build-and-apply-naive-bayes-classification-for-spam-filtering-2b8d3308501

Dataset from https://archive.ics.uci.edu/ml/datasets/sms+spam+collection



---



Download and Extract the Required Dataset

In [ ]:
%%bash
wget -qO dataset.zip https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip
unzip -qo dataset.zip
rm dataset.zip readme



---



In [ ]:
import pandas as pd

In [ ]:
# Line #5082 in the dataset has an unbalanced double quote causing read_csv() to merge 3 samples into one.
# By using quoting=3 make the function ignores the double quote.
sms_data = pd.read_csv('SMSSpamCollection', header=None, sep='\t', names=['Label', 'SMS'], quoting=3)

In [ ]:
sms_data.head()

,Label,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
sms_data.shape

(5574, 2)

In [ ]:
sms_data.loc[30,:]

Label                                                  ham
SMS      Wait that's still not all that clear, were you...
Name: 30, dtype: object

In [ ]:
sms_data.loc[30,'SMS']

"Wait that's still not all that clear, were you not sure about me being sarcastic or that that's why x doesn't want to live with us"

In [ ]:
sms_data.groupby('Label').count()

,SMS
Label,
ham,4827
spam,747


Prepare data

In [ ]:
sms_data_clean = sms_data.copy()

In [ ]:
# Use regex=True in replace() to fix the FutureWarning
sms_data_clean['SMS'] = sms_data_clean['SMS'].str.replace('\W+', ' ', regex=True).str.replace('\s+', ' ', regex=True).str.strip()

In [ ]:
sms_data_clean['SMS'] = sms_data_clean['SMS'].str.lower()

In [ ]:
sms_data_clean['SMS'] = sms_data_clean['SMS'].str.split()

In [ ]:
sms_data_clean['SMS'].head()

0    [go, until, jurong, point, crazy, available, o...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, in, 2, a, wkly, comp, to, win, f...
3    [u, dun, say, so, early, hor, u, c, already, t...
4    [nah, i, don, t, think, he, goes, to, usf, he,...
Name: SMS, dtype: object

In [ ]:
sms_data.loc[2,'SMS']

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"

In [ ]:
sms_data_clean.loc[2,'SMS']

['free',
 'entry',
 'in',
 '2',
 'a',
 'wkly',
 'comp',
 'to',
 'win',
 'fa',
 'cup',
 'final',
 'tkts',
 '21st',
 'may',
 '2005',
 'text',
 'fa',
 'to',
 '87121',
 'to',
 'receive',
 'entry',
 'question',
 'std',
 'txt',
 'rate',
 't',
 'c',
 's',
 'apply',
 '08452810075over18',
 's']

In [ ]:
sms_data_clean['Label'].value_counts() / sms_data.shape[0] * 100

ham     86.598493
spam    13.401507
Name: Label, dtype: float64

Split to train and test data

In [ ]:
# shouldn't 1st line doesn't need to reset_index() just yet? because 2nd line needs its indices to drop the selected training samples.
train_data = sms_data_clean.sample(frac=0.2,random_state=1)#.reset_index(drop=True)
test_data = sms_data_clean.drop(train_data.index).reset_index(drop=True)
train_data = train_data.reset_index(drop=True)

In [ ]:
train_data['Label'].value_counts() / train_data.shape[0] * 100

ham     87.623318
spam    12.376682
Name: Label, dtype: float64

In [ ]:
train_data.shape

(1115, 2)

In [ ]:
test_data['Label'].value_counts() / test_data.shape[0] * 100

ham     86.342229
spam    13.657771
Name: Label, dtype: float64

In [ ]:
test_data.shape

(4459, 2)

In [ ]:
test_data.head()

,Label,SMS
0,ham,"[go, until, jurong, point, crazy, available, o..."
1,ham,"[ok, lar, joking, wif, u, oni]"
2,spam,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,ham,"[u, dun, say, so, early, hor, u, c, already, t..."
4,ham,"[even, my, brother, is, not, like, to, speak, ..."


Prepare vocabulary - the list fo all the words from the dataset

In [ ]:
vocabulary = list(set(train_data['SMS'].sum()))

In [ ]:
vocabulary[11:20]

['seems', 'wrld', 'back', 'tiger', 'trains', 'sunny', 'await', 'nicky', 'hate']

In [ ]:
len(vocabulary)

3589

Calculate frequencies fo the words for each message

In [ ]:
word_counts_per_sms = pd.DataFrame([
    [row[1].count(word) for word in vocabulary]
    for _, row in train_data.iterrows()], columns=vocabulary)

In [ ]:
train_data = pd.concat([train_data.reset_index(), word_counts_per_sms], axis=1).iloc[:,1:]

In [ ]:
train_data.head()

,Label,SMS,files,repent,xchat,regret,working,2nhite,talks,apply,...,asap,diamond,something,careful,ans,affair,pthis,gimmi,excellent,thinks
0,ham,"[looks, like, u, wil, b, getting, a, headstart...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,ham,"[i, noe, la, u, wana, pei, bf, oso, rite, k, l...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ham,"[2mro, i, am, not, coming, to, gym, machan, go...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,spam,"[todays, vodafone, numbers, ending, with, 4882...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ham,"[hi, hope, ur, day, good, back, from, walk, ta...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
train_data['i']

0       1
1       1
2       1
3       0
4       0
       ..
1110    0
1111    0
1112    0
1113    0
1114    0
Name: i, Length: 1115, dtype: int64

Calculate values for the Bayes formula

In [ ]:
alpha = 1

In [ ]:
Nvoc = len(train_data.columns) - 3

In [ ]:
Pspam = train_data['Label'].value_counts()['spam'] / train_data.shape[0]

In [ ]:
Pham = train_data['Label'].value_counts()['ham'] / train_data.shape[0]

In [ ]:
Nspam = train_data.loc[train_data['Label'] == 'spam', 'SMS'].apply(len).sum()

In [ ]:
Nham = train_data.loc[train_data['Label'] == 'ham', 'SMS'].apply(len).sum()

In [ ]:
Nspam

3530

In [ ]:
Nham

14605

In [ ]:
def p_w_spam(word):
    if word in train_data.columns:
        return (train_data.loc[train_data['Label'] == 'spam', word].sum() + alpha) / (Nspam + alpha*Nvoc)
    else:
        return 1

In [ ]:
def p_w_ham(word):
    if word in train_data.columns:
        return (train_data.loc[train_data['Label'] == 'ham', word].sum() + alpha) / (Nham + alpha*Nvoc)
    else:
        return 1

In [ ]:
def classify(message):
    p_spam_given_message = Pspam
    p_ham_given_message = Pham
    for word in message:
        p_spam_given_message *= p_w_spam(word)
        p_ham_given_message *= p_w_ham(word)
    if p_ham_given_message > p_spam_given_message:
        return 'ham'
    elif p_ham_given_message < p_spam_given_message:
        return 'spam'
    else:
        return 'needs human classification'

In [ ]:
classify('secret')

'ham'

In [ ]:
classify(['secret', 'source', 'of', 'infinite', 'power'])

'ham'

In [ ]:
classify(['free', 'win', 'welcom'])

'spam'

Test with some messages

In [75]:
classify('congratulations you ve just got a reward ticket call 555-123456 for the codes'.split())

'ham'

In [79]:
classify('hello mom are you free can i get some money to buy some cheap candy at the nearby store'.split())

'ham'

Test data

In [ ]:
test_data['predicted'] = test_data['SMS'].apply(classify)

In [ ]:
test_data.head()

,Label,SMS,predicted
0,ham,"[go, until, jurong, point, crazy, available, o...",ham
1,ham,"[ok, lar, joking, wif, u, oni]",ham
2,spam,"[free, entry, in, 2, a, wkly, comp, to, win, f...",spam
3,ham,"[u, dun, say, so, early, hor, u, c, already, t...",ham
4,ham,"[even, my, brother, is, not, like, to, speak, ...",ham


In [ ]:
correct = (test_data['predicted'] == test_data['Label']).sum() / test_data.shape[0] * 100

In [ ]:
test_data.loc[test_data['predicted'] != test_data['Label']]

,Label,SMS,predicted
52,spam,"[did, you, hear, about, the, new, divorce, bar...",ham
176,spam,"[will, u, meet, ur, dream, partner, soon, is, ...",ham
421,spam,"[you, will, recieve, your, tone, within, the, ...",ham
519,spam,"[88800, and, 89034, are, premium, phone, servi...",ham
529,spam,"[sms, ac, sun0819, posts, hello, you, seem, co...",ham
...,...,...,...
4025,spam,"[i, d, like, to, tell, you, my, deepest, darke...",ham
4037,ham,"[we, have, sent, jd, for, customer, service, c...",spam
4305,spam,"[dating, i, have, had, two, of, these, only, s...",ham
4311,spam,"[the, current, leading, bid, is, 151, to, paus...",ham


In [ ]:
correct

98.04888988562458

## Assignment From Slides

### Table 1

\begin{array}{|c|c|c|c|c|}
\hline
                         & \textbf{Spam} & \textbf{Spam (ratio)} & \textbf{No Spam} & \textbf{No Spam (ratio)} \\ \hline
\textbf{Total}           & 25            & 1                     & 75               & 1                        \\ \hline
Buy\ (B)                      & 20            & \frac{4}{5}                   & 5                & \frac{1}{15}                     \\ \hline
Cheap\ (C)                    & 15            & \frac{3}{5}                   & 10               & \frac{2}{15}                     \\ \hline
Work\ (W)                     & 5             & \frac{1}{5}                   & 30               & \frac{6}{15}                     \\ \hline
Free\ (F)                     & 20            & \frac{4}{5}                   & 7                & \frac{7}{75}                     \\ \hline
Buy, Cheap, Work\ \&\ Free & \frac{48}{25}         & \frac{48}{625}                & \frac{28}{1125}          & \frac{28}{84375}                 \\ \hline
\end{array}

$$
P(S\ |\ B\cap C\cap W\cap F) = \\
\frac{P(B|S)P(C|S)P(W|S)P(F|S)P(S)}{P(B|S)P(C|S)P(W|S)P(F|S)P(S)+P(B|H)P(C|H)P(W|H)P(F|H)P(H)} = \\
\frac{\frac{20}{25}\frac{15}{25}\frac{5}{25}\frac{20}{25}\frac{25}{100}}{\frac{20}{25}\frac{15}{25}\frac{5}{25}\frac{20}{25}\frac{25}{100}+\frac{5}{75}\frac{10}{75}\frac{30}{75}\frac{7}{75}\frac{75}{100}} = \frac{540}{547} = 98.72\%
$$

### Table 2

\begin{array}{|c|c|c|c|c|}
\hline
                         & \textbf{Spam} & \textbf{Spam (ratio)} & \textbf{No Spam} & \textbf{No Spam (ratio)} \\ \hline
\textbf{Total}           & 25            & 1                     & 75               & 1                        \\ \hline
Buy\ (B)                      & 20            & \frac{4}{5}                   & 5                & \frac{1}{15}                     \\ \hline
Cheap\ (C)                    & 15            & \frac{3}{5}                   & 10               & \frac{2}{15}                     \\ \hline
Work\ (W)                     & 5             & \frac{1}{5}                   & 30               & \frac{6}{15}                     \\ \hline
Free\ (F)                     & 20            & \frac{4}{5}                   & 7                & \frac{7}{75}                     \\ \hline
Will\ (U)                     & 4             & \frac{4}{25}                  & 40               & \frac{8}{15}                     \\ \hline
Buy, Cheap, Work, Free\ \&\ Will & \frac{192}{625}       & \frac{192}{15625}             & \frac{224}{16875}        & \frac{224}{1265625}              \\ \hline
\end{array}

$$
P(S\ |\ B\cap C\cap W\cap F\cap U) = \\
\frac{P(B|S)P(C|S)P(W|S)P(F|S)P(U|S)P(S)}{P(B|S)P(C|S)P(W|S)P(F|S)P(U|S)P(S)+P(B|H)P(C|H)P(W|H)P(F|H)P(U|H)P(H)} = \\
\frac{\frac{20}{25}\frac{15}{25}\frac{5}{25}\frac{20}{25}\frac{4}{25}\frac{25}{100}}{\frac{20}{25}\frac{15}{25}\frac{5}{25}\frac{20}{25}\frac{4}{25}\frac{25}{100}+\frac{5}{75}\frac{10}{75}\frac{30}{75}\frac{7}{75}\frac{40}{75}\frac{75}{100}} = \frac{162}{169} = 95.86\%
$$